## Current ML - Corrector

In [ ]:
# Cell 5 (REPLACED): Hybrid ML-corrector (LSTM corrector on residuals)
# Uses a stabilized lstm_paper_loss_hybrid to avoid NaNs.

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# ---------- Build training data where target = residual (true - bergman_pred) ----------
X_corr = []
y_corr = []
for sid, info in per_subject_train.items():
    Xs = info['X']   # scaled inputs
    if Xs.shape[0] == 0:
        continue
    subj = per_subject[sid]
    L = subj.shape[0]
    for i in range(N_IN, L - N_OUT + 1):
        xin = Xs[i - N_IN]  # input sequence aligned with create_sequences indexing
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1,1)
        y_true_scaled = cg_scaler.transform(subj['CGM_smoothed'].values[i:i+N_OUT].reshape(-1,1)).reshape(-1,1)
        resid = (y_true_scaled - berg_pred_scaled).reshape(N_OUT)  # flattened n_out
        X_corr.append(xin)
        y_corr.append(resid)
if len(X_corr) == 0:
    raise RuntimeError("No training data found for ML-corrector.")
X_corr = np.stack(X_corr).astype(np.float32)  # (N_samples, n_in, n_feats)
y_corr = np.stack(y_corr).astype(np.float32)  # (N_samples, n_out)

n_in = X_corr.shape[1]
n_feats = X_corr.shape[2]
n_out = y_corr.shape[1]

# ---------- Stabilized lstm_paper_loss_hybrid ----------
# constants from paper
C = 32.9170
w_left = 19.0
w_right = 1.0
T_mgdl = 105.0  # target BG in mg/dL

# slope-cost constants (tuneable)
a = 1.0
b = 1.0

eps = 1e-6   # slightly larger eps for numerical safety

min_v = float(cg_scaler.data_min_[0])
max_v = float(cg_scaler.data_max_[0])
scale_range = max_v - min_v if (max_v - min_v) != 0 else 1.0

def _to_mgdl(tensor_scaled):
    """Inverse min-max scaling, clip to safe positive values to avoid log(0)."""
    mgdl = tensor_scaled * scale_range + min_v
    # clip to >= small positive to avoid log/zero issues
    return tf.clip_by_value(mgdl, 1e-3, 1000.0)  # upper clip to avoid extreme outliers

def _zone_cost(bg_mgdl):
    """Stable zone cost using log difference squared with small eps added."""
    bg = tf.cast(bg_mgdl, tf.float32)
    log_bg = tf.math.log(bg + eps)
    log_T = tf.math.log(tf.cast(T_mgdl, tf.float32) + eps)
    diff2 = tf.square(log_bg - log_T)
    left_mask = tf.cast(bg < T_mgdl, dtype=bg.dtype)
    right_mask = 1.0 - left_mask
    cost = C * (w_left * diff2 * left_mask + w_right * diff2 * right_mask)
    return cost

def _slope_cost(delta_bg_mgdl):
    """Slope cost; convert mg/dL -> mmol/L safely and apply piecewise weights."""
    delta = tf.cast(delta_bg_mgdl, tf.float32) / 18.0
    neg_mask = tf.cast(delta < 0.0, dtype=delta.dtype)
    pos_mask = 1.0 - neg_mask
    cost = (2.0 * b) * tf.square(delta) * neg_mask + a * tf.square(delta) * pos_mask
    return cost

alpha = 0.6  # L1 weight (can tune)

def lstm_paper_loss_hybrid(y_true, y_pred):
    """
    y_true / y_pred are in scaled units [0,1] shape (batch, n_out).
    Loss computed in mg/dL with stabilized operations and normalized weights.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # convert to mg/dL and clip
    y_true_mg = _to_mgdl(y_true)
    y_pred_mg = _to_mgdl(y_pred)

    # compute delta (slope) from true in mg/dL (first delta is 0)
    y_true_shift = tf.concat([y_true_mg[:, :1] * 0.0, y_true_mg[:, :-1]], axis=1)
    delta_true = y_true_mg - y_true_shift

    # zone and slope cost
    zone = _zone_cost(y_true_mg)           # (batch, n_out)
    slope = _slope_cost(delta_true)       # (batch, n_out)
    weights = zone + slope + 1.0          # baseline +1

    # normalize weights to avoid extremely large multipliers
    mean_w = tf.reduce_mean(weights)
    weights_norm = weights / (mean_w + 1e-6)

    # use squared error in mg/dL (consistent units)
    se_mg = tf.square(y_pred_mg - y_true_mg)
    weighted_se = weights_norm * se_mg
    mse_weighted = tf.reduce_mean(weighted_se)

    # L1 in mg/dL (normalized by scale_range to keep comparable magnitude)
    l1_mg = tf.reduce_mean(tf.abs(y_pred_mg - y_true_mg)) / (scale_range + 1e-6)

    loss = mse_weighted + alpha * l1_mg
    # final safety clip to avoid NaN/Inf
    loss = tf.where(tf.math.is_finite(loss), loss, tf.constant(1e6, dtype=loss.dtype))
    return loss

# ---------- Training model (stable optimizer, gradient clipping) ----------
tf.keras.backend.clear_session()
model_corr = Sequential([
    LSTM(32, return_sequences=False, input_shape=(n_in, n_feats)),
    Dense(n_out, activation='linear')
])

opt = Adam(learning_rate=1e-4, clipnorm=1.0)   # smaller lr + clipnorm for stability
model_corr.compile(optimizer=opt, loss='mse')
print(model_corr.summary())

# Fit: we can add a small validation_split to monitor val loss if desired
history = model_corr.fit(X_corr, y_corr, epochs=20, batch_size=64, verbose=2, validation_split=0.05)

# ---------- Test-time predictor using corrected model ----------
def ml_corrector_predict_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, N_OUT, 1))
    subj = per_subject[sid]
    L = subj.shape[0]
    preds_all = []
    for i in range(N_IN, L - N_OUT + 1):
        xin = info['X'][i - N_IN]  # aligned input
        xin_batch = xin.reshape(1, n_in, n_feats)
        resid_pred = model_corr.predict(xin_batch)[0]  # shape (n_out,)
        # bergman pred for this window
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1)
        corrected = berg_pred_scaled + resid_pred
        preds_all.append(corrected.reshape(-1,1))
    if len(preds_all) == 0:
        return np.zeros((0,N_OUT,1))
    return np.stack(preds_all)

ml_corrector_model = {"lstm_corrector": model_corr, "predict": ml_corrector_predict_subject}
